# Practical session: Continuous BMA-EM step by step

This notebook is designed for a small hands-on practical on **Bayesian Model Averaging with EM** for a **continuous** target.

## Learning goals
By the end, students should be able to:
- compare **simple model averaging (SMA)** and **BMA-EM**,
- understand how **weights emerge from the data**,
- compute **within-model** and **between-model** variance,
- see why **spatial generalization** is difficult.

The dataset is synthetic on purpose: it was built so that the models have different strengths and weaknesses.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Display options
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

## 1. Load the dataset

The table contains:
- a reference variable: `y_true`
- predictions from four models
- spatial coordinates
- a few environmental predictors
- a predefined spatial split (`train` / `test`)

In [2]:
df = pd.read_csv("bma_em_practical_dataset.csv")
df.head()

,site_id,lat,lon,temp_c,precip_mm,elev_m,wetness_index,forest_frac,y_true,model_1_biased_lowvar,model_2_unbiased_noisy,model_3_context_dependent,model_4_bad_spatial,split
0,S001,-6.3098,39.1025,20.023,1045.2,209.0,0.772,0.803,201.082,177.539,209.778,201.646,129.133,train
1,S002,68.5929,166.6183,4.971,1020.2,724.9,0.325,0.315,110.592,103.464,100.217,84.400,101.856,test
2,S003,40.1592,-122.3714,11.692,701.8,407.8,0.458,0.465,143.105,126.094,163.450,130.185,118.031,train
3,S004,22.8256,6.2321,20.734,715.5,786.7,0.426,0.626,181.828,164.945,182.353,185.433,129.643,train
4,S005,-34.7176,128.3068,10.341,633.3,513.2,0.341,0.597,149.193,140.958,146.123,133.215,85.320,train


In [3]:
print(df.shape)
print(df["split"].value_counts())

(260, 14)
split
train    200
test      60
Name: count, dtype: int64


## 2. Define the model matrix and a few helper functions

In [4]:
model_cols = [
    "model_1_biased_lowvar",
    "model_2_unbiased_noisy",
    "model_3_context_dependent",
    "model_4_bad_spatial",
]

X = df[model_cols].to_numpy()
y = df["y_true"].to_numpy()

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def r2_score_manual(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot

def summarize_performance(name, y_true, y_pred):
    return {
        "method": name,
        "RMSE": rmse(y_true, y_pred),
        "R2": r2_score_manual(y_true, y_pred),
    }

## 3. First look at the individual models

A good first exercise is simply to compare the four models.

In [5]:
results = []
for col in model_cols:
    results.append(summarize_performance(col, y, df[col].to_numpy()))

pd.DataFrame(results).sort_values("RMSE")

,method,RMSE,R2
1,model_2_unbiased_noisy,11.425326,0.891931
2,model_3_context_dependent,13.208373,0.855569
0,model_1_biased_lowvar,17.405885,0.749184
3,model_4_bad_spatial,55.815748,-1.579146


In [ ]:
for col in model_cols:
    plt.figure(figsize=(5, 5))
    plt.scatter(df["y_true"], df[col], alpha=0.7)
    mn = min(df["y_true"].min(), df[col].min())
    mx = max(df["y_true"].max(), df[col].max())
    plt.plot([mn, mx], [mn, mx])
    plt.xlabel("Observed")
    plt.ylabel(col)
    plt.title(col)
    plt.show()

## 4. Simple model averaging (SMA)

This is the natural baseline:
\[
\hat{y}_{SMA} = \frac{1}{M} \sum_{k=1}^M f_k
\]

In [ ]:
y_sma = X.mean(axis=1)
pd.DataFrame([summarize_performance("SMA", y, y_sma)])

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(y, y_sma, alpha=0.7)
mn = min(y.min(), y_sma.min())
mx = max(y.max(), y_sma.max())
plt.plot([mn, mx], [mn, mx])
plt.xlabel("Observed")
plt.ylabel("SMA prediction")
plt.title("Simple model averaging")
plt.show()

## 5. Implement BMA-EM

We use the usual continuous formulation:

\[
p(y_i \mid \text{models}) = \sum_{k=1}^M w_k \, \mathcal{N}(y_i \mid f_{ik}, \sigma_k^2)
\]

where:
- \(w_k\) are non-negative weights summing to 1,
- \(f_{ik}\) is the prediction of model \(k\) for sample \(i\),
- \(\sigma_k^2\) is a model-specific variance term.

### EM logic
- **E-step**: compute responsibilities
- **M-step**: update weights and sigmas

In [ ]:
def fit_bma_em(
    X,
    y,
    max_iter=500,
    ll_tol=1e-6,
    sigma_floor=1e-6,
    denom_floor=1e-12,
    weight_floor=1e-12,
    verbose=True,
):
    '''
    Fit continuous BMA-EM where each model prediction is the mean of a Gaussian component.
    X: shape (n_samples, n_models)
    y: shape (n_samples,)
    '''
    n, m = X.shape

    # Initialize weights equally
    weights = np.ones(m) / m

    # Initialize sigmas from each model RMSE
    sigmas = np.sqrt(np.mean((X - y[:, None]) ** 2, axis=0))
    sigmas = np.maximum(sigmas, sigma_floor)

    loglik_trace = []

    for it in range(max_iter):
        # --------------------------------------------------
        # E-step: responsibilities
        # r_ik proportional to w_k * N(y_i | X_ik, sigma_k^2)
        # --------------------------------------------------
        diff = y[:, None] - X
        gaussian = (
            1.0 / (np.sqrt(2 * np.pi) * sigmas[None, :])
            * np.exp(-0.5 * (diff / sigmas[None, :]) ** 2)
        )
        numerators = weights[None, :] * gaussian
        denominators = np.sum(numerators, axis=1, keepdims=True)
        denominators = np.maximum(denominators, denom_floor)
        responsibilities = numerators / denominators

        # --------------------------------------------------
        # M-step: weights and sigmas
        # --------------------------------------------------
        Nk = responsibilities.sum(axis=0)
        weights_new = Nk / n
        weights_new = np.maximum(weights_new, weight_floor)
        weights_new = weights_new / weights_new.sum()

        sigmas_new = np.sqrt(
            np.sum(responsibilities * (diff ** 2), axis=0) / np.maximum(Nk, denom_floor)
        )
        sigmas_new = np.maximum(sigmas_new, sigma_floor)

        # Log-likelihood
        gaussian_new = (
            1.0 / (np.sqrt(2 * np.pi) * sigmas_new[None, :])
            * np.exp(-0.5 * (diff / sigmas_new[None, :]) ** 2)
        )
        mixture = np.sum(weights_new[None, :] * gaussian_new, axis=1)
        mixture = np.maximum(mixture, denom_floor)
        ll = np.sum(np.log(mixture))
        loglik_trace.append(ll)

        if verbose and (it < 5 or (it + 1) % 20 == 0):
            print(f"iter={it+1:03d}  loglik={ll:.6f}")

        # Convergence check
        if it > 0 and abs(loglik_trace[-1] - loglik_trace[-2]) < ll_tol:
            weights = weights_new
            sigmas = sigmas_new
            break

        weights = weights_new
        sigmas = sigmas_new

    return {
        "weights": weights,
        "sigmas": sigmas,
        "responsibilities": responsibilities,
        "loglik_trace": np.array(loglik_trace),
    }

def predict_bma_mean(X, weights):
    return X @ weights

def predict_bma_variance(X, weights, sigmas):
    '''
    Total predictive variance decomposition:
    total = within + between
    '''
    mean_pred = predict_bma_mean(X, weights)
    within = np.sum(weights * (sigmas ** 2))
    between = np.sum(weights[None, :] * (X - mean_pred[:, None]) ** 2, axis=1)
    total = within + between
    return mean_pred, within, between, total

## 6. Fit BMA-EM on the full dataset first

This is the simplest case: just to understand the mechanics.

In [ ]:
fit_all = fit_bma_em(X, y, verbose=False)
weights_all = fit_all["weights"]
sigmas_all = fit_all["sigmas"]

pd.DataFrame({
    "model": model_cols,
    "weight": weights_all,
    "sigma": sigmas_all,
}).sort_values("weight", ascending=False)

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(fit_all["loglik_trace"])
plt.xlabel("Iteration")
plt.ylabel("Log-likelihood")
plt.title("EM convergence")
plt.show()

In [ ]:
y_bma_all, within_all, between_all, total_all = predict_bma_variance(X, weights_all, sigmas_all)

comparison = pd.DataFrame([
    summarize_performance("SMA", y, y_sma),
    summarize_performance("BMA-EM", y, y_bma_all),
]).sort_values("RMSE")

comparison

### Interpretation questions
- Which model got the largest weight?
- Which model got close to zero?
- Does BMA-EM improve over SMA?
- When might BMA become close to equal weights?

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(y, y_bma_all, alpha=0.7)
mn = min(y.min(), y_bma_all.min())
mx = max(y.max(), y_bma_all.max())
plt.plot([mn, mx], [mn, mx])
plt.xlabel("Observed")
plt.ylabel("BMA-EM prediction")
plt.title("BMA-EM on all data")
plt.show()

## 7. Variance decomposition

For the mixture mean:
\[
\bar{f}_i = \sum_k w_k f_{ik}
\]

The predictive variance can be decomposed as:

- **within-model variance**
\[
\sum_k w_k \sigma_k^2
\]

- **between-model variance**
\[
\sum_k w_k (f_{ik} - \bar{f}_i)^2
\]

In this simple formulation, the within term is constant across samples, while the between term varies from one sample to another.

In [ ]:
print("Within-model variance (constant here):", within_all)
print("Mean between-model variance:", np.mean(between_all))
print("Mean total predictive variance:", np.mean(total_all))

In [ ]:
variance_df = pd.DataFrame({
    "site_id": df["site_id"],
    "lat": df["lat"],
    "lon": df["lon"],
    "within_variance": within_all,
    "between_variance": between_all,
    "total_variance": total_all,
})
variance_df.head()

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(between_all, bins=25)
plt.xlabel("Between-model variance")
plt.ylabel("Count")
plt.title("Distribution of between-model variance")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(df["lon"], between_all, alpha=0.7)
plt.xlabel("Longitude")
plt.ylabel("Between-model variance")
plt.title("Where do models disagree more?")
plt.show()

## 8. Spatial generalization: train on one region, test on another

Now we move to the key conceptual point.

We fit BMA-EM **only on the training subset**, then evaluate on the held-out spatial subset.

In [ ]:
train_mask = df["split"] == "train"
test_mask = df["split"] == "test"

X_train = df.loc[train_mask, model_cols].to_numpy()
y_train = df.loc[train_mask, "y_true"].to_numpy()

X_test = df.loc[test_mask, model_cols].to_numpy()
y_test = df.loc[test_mask, "y_true"].to_numpy()

fit_train = fit_bma_em(X_train, y_train, verbose=False)
weights_train = fit_train["weights"]
sigmas_train = fit_train["sigmas"]

pd.DataFrame({
    "model": model_cols,
    "weight_train_only": weights_train,
    "sigma_train_only": sigmas_train,
}).sort_values("weight_train_only", ascending=False)

In [ ]:
# Predictions on train and test
y_sma_train = X_train.mean(axis=1)
y_sma_test = X_test.mean(axis=1)

y_bma_train, within_train, between_train, total_train = predict_bma_variance(X_train, weights_train, sigmas_train)
y_bma_test, within_test, between_test, total_test = predict_bma_variance(X_test, weights_train, sigmas_train)

perf_spatial = pd.DataFrame([
    {"subset": "train", **summarize_performance("SMA", y_train, y_sma_train)},
    {"subset": "train", **summarize_performance("BMA-EM", y_train, y_bma_train)},
    {"subset": "test", **summarize_performance("SMA", y_test, y_sma_test)},
    {"subset": "test", **summarize_performance("BMA-EM", y_test, y_bma_test)},
])

perf_spatial

### Discussion
This is usually where the most interesting conversation starts.

Questions:
- Does BMA still help on the test region?
- Are the learned weights still sensible elsewhere?
- Why might weights be **local** rather than universal?

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(y_test, y_sma_test, alpha=0.7, label="SMA")
plt.scatter(y_test, y_bma_test, alpha=0.7, label="BMA-EM")
mn = min(y_test.min(), y_sma_test.min(), y_bma_test.min())
mx = max(y_test.max(), y_sma_test.max(), y_bma_test.max())
plt.plot([mn, mx], [mn, mx])
plt.xlabel("Observed (test)")
plt.ylabel("Prediction")
plt.title("Held-out spatial test set")
plt.legend()
plt.show()

## 9. Optional extension: local behavior by environmental regime

Here we inspect whether some models look better in specific contexts.

In [ ]:
df["wet_regime"] = np.where(df["wetness_index"] > df["wetness_index"].median(), "wet", "dry")

group_rows = []
for regime_name, sub in df.groupby("wet_regime"):
    for col in model_cols:
        group_rows.append({
            "regime": regime_name,
            "model": col,
            "RMSE": rmse(sub["y_true"].to_numpy(), sub[col].to_numpy()),
            "R2": r2_score_manual(sub["y_true"].to_numpy(), sub[col].to_numpy()),
        })

pd.DataFrame(group_rows).sort_values(["regime", "RMSE"])

This often reveals a crucial point:

> a model can be weak globally but useful in a specific environmental context.

That is exactly why **fixed global weights** can be too simplistic in real spatial applications.

## 10. Small exercises for students

### Exercise A
Change the initialization of the weights and check whether EM converges to a similar solution.

### Exercise B
Remove the worst model and rerun SMA and BMA-EM.

### Exercise C
Create a random train/test split instead of the spatial split.  
How different are the conclusions?

### Exercise D
Try to estimate **site-specific** or **cluster-specific** weights by splitting the data into two environmental groups.

In [ ]:
# Example template for Exercise C
# Uncomment and complete if you want the students to try it.

# rng = np.random.default_rng(0)
# idx = np.arange(len(df))
# rng.shuffle(idx)
# n_train = int(0.7 * len(df))
# train_idx = idx[:n_train]
# test_idx = idx[n_train:]

## 11. Final message

The important conceptual takeaway is not only that **BMA can improve predictions**, but that:

- weights are **learned from data**,
- uncertainty has a **within / between** decomposition,
- and **spatial transferability of weights is not guaranteed**.

That last point is often the bridge toward clustering, environmental domains, or spatial aggregation strategies.